# Combinação controlada P1+A1

Este notebook investiga se o Shades-of-Gray (`P1`) e o augmentation geométrico (`A1`) têm efeitos complementares sobre a DeepResUNet. O modelo começa do zero. Teste interno e FUSeg permanecem fechados.

## O que muda e o que permanece fixo

No treino: imagem original → P1 → resize/pad → A1 → tensor `[0,1]`. Na validação: imagem original → P1 → resize/pad → tensor `[0,1]`, sem augmentation. Arquitetura, split, seed 42, resolução, loss, otimizador, scheduler, early stopping e regra de threshold são os mesmos do D0.

## Interpretação planejada

- se P1+A1 superar P1 e A1, há evidência de complementaridade nesta seed;
- se ficar próximo do melhor isolado, os efeitos podem ser redundantes;
- se piorar, a combinação pode interferir negativamente.

A comparação será feita somente depois de retornar os artefatos.

## 1. Montar o Drive e localizar o projeto

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, yaml

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Execução fora do Colab.")

candidates = [
    Path("/content/drive/MyDrive/TCC/Projeto_TCC_Lucas"),
    Path("/content/drive/MyDrive/Projeto_TCC_Lucas"),
]
PROJECT_ROOT = next((p for p in candidates if p.is_dir()), Path.cwd())
SOURCE_ROOT = PROJECT_ROOT / "src"
os.chdir(PROJECT_ROOT)
os.environ["PYTHONPATH"] = str(SOURCE_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
print("Projeto:", PROJECT_ROOT)

## 2. Instalar dependências e preparar o cache temporário

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()
DATA_ROOT = Path("/content/tcc_data") if Path("/content").is_dir() else PROJECT_ROOT
if DATA_ROOT != PROJECT_ROOT:
    shutil.copytree(PROJECT_ROOT / "Dataset", DATA_ROOT / "Dataset", dirs_exist_ok=True)
print("Dados:", DATA_ROOT / "Dataset")

## 3. Validar que a configuração combina exatamente P1 e A1

In [ ]:
from IPython.display import display

CONFIG_PATH = PROJECT_ROOT / "configs/combined_preprocessing_augmentation_p1_a1.yaml"
CFG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
D0 = yaml.safe_load((PROJECT_ROOT / "configs/baseline_d0.yaml").read_text(encoding="utf-8"))
P1 = yaml.safe_load(
    (PROJECT_ROOT / "configs/preprocessing_dp1_shades_of_gray.yaml").read_text(encoding="utf-8")
)
A1 = yaml.safe_load((PROJECT_ROOT / "configs/augmentation_d0_a1.yaml").read_text(encoding="utf-8"))
AUDIT = yaml.safe_load(
    (PROJECT_ROOT / "configs/augmentation_audit.yaml").read_text(encoding="utf-8")
)
from tcc_segmentation.combined_preprocessing_augmentation import validate_p1_a1_config

validate_p1_a1_config(CFG, d0_config=D0, p1_config=P1, a1_config=A1, audit_config=AUDIT)
display(
    {
        "experimento": CFG["experiment_id"],
        "pré-processamento": CFG["data"]["color_preprocessing"],
        "p": CFG["data"]["color_preprocessing_parameters"]["minkowski_p"],
        "augmentation": CFG["data"]["augmentation"],
        "pesos prévios": CFG["model"]["pretrained"],
        "seed": CFG["seed"],
        "contraste primário": CFG["comparison_plan"]["primary_contrast"],
        "teste permitido": CFG["comparison_plan"]["test_access_allowed"],
    }
)

## 4. Executar os testes técnicos

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(SOURCE_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_combined_preprocessing_augmentation.py",
        "tests/test_augmentation.py",
        "tests/test_preprocessing.py",
    ],
    cwd=PROJECT_ROOT,
    env=env,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

## 5. Visualizar a ordem real do pipeline

A linha superior mostra P1+A1 no treino; a inferior mostra P1 na validação. A máscara deve permanecer binária e alinhada.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tcc_segmentation.data import DevelopmentDataset
from tcc_segmentation.combined_preprocessing_augmentation import (
    make_p1_a1_train_transform,
    make_p1_validation_transform,
)

train_transform = make_p1_a1_train_transform(CFG)
train_transform.set_epoch(0)
train_ds = DevelopmentDataset(
    PROJECT_ROOT / CFG["data"]["train_manifest"],
    DATA_ROOT,
    "train",
    transform=train_transform,
    mask_threshold=CFG["data"]["mask_threshold"],
)
val_ds = DevelopmentDataset(
    PROJECT_ROOT / CFG["data"]["validation_manifest"],
    DATA_ROOT,
    "validation",
    transform=make_p1_validation_transform(CFG),
    mask_threshold=CFG["data"]["mask_threshold"],
)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, (label, ds) in enumerate([("treino: P1+A1", train_ds), ("validação: somente P1", val_ds)]):
    for col in range(4):
        sample = ds[col]
        image = sample["image"].transpose(1, 2, 0)
        mask = sample["mask"][0]
        axes[row, col].imshow(image)
        axes[row, col].contour(mask, levels=[0.5], colors=["lime"], linewidths=1)
        axes[row, col].set_title(label + " | " + sample["canonical_id"])
        axes[row, col].axis("off")
        assert set(np.unique(mask)).issubset({0.0, 1.0})
plt.tight_layout()
plt.show()

## 6. Função para acompanhar cada época

In [ ]:
def run_stream(command):
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    returncode = process.wait()
    print("\nreturncode=", returncode)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)

## 7. Treinar P1+A1

Mantenha `RESUME_DIR=None` para uma execução nova. Em caso de desconexão, informe o diretório criado para continuar a mesma execução. Não aponte para diretórios de D0, P1 ou A1.

In [ ]:
# Exemplo: RESUME_DIR=PROJECT_ROOT/'results/day8/P1A1/P1A1_seed42_...'
RESUME_DIR = None
command = [
    sys.executable,
    "scripts/run_combined_preprocessing_augmentation.py",
    "--project-root",
    str(PROJECT_ROOT),
    "--data-root",
    str(DATA_ROOT),
    "--config",
    str(CONFIG_PATH),
]
if RESUME_DIR is not None:
    command += ["--resume-dir", str(RESUME_DIR)]
run_stream(command)

## 8. Localizar e visualizar as saídas

In [ ]:
from PIL import Image
import pandas as pd

OUT = (
    Path(RESUME_DIR)
    if RESUME_DIR
    else max(
        (PROJECT_ROOT / CFG["runtime"]["output_root"]).iterdir(), key=lambda p: p.stat().st_mtime
    )
)
SUMMARY = json.loads((OUT / "final_summary.json").read_text(encoding="utf-8"))
SAFETY = json.loads((OUT / "augmentation_safety.json").read_text(encoding="utf-8"))
display(
    {
        "run": SUMMARY["run_id"],
        "melhor época": SUMMARY["training"]["best_epoch"],
        "Dice 0,5": SUMMARY["selection"]["metrics_at_0_5"]["macro"]["dice"]["mean"],
        "threshold": SUMMARY["selection"]["threshold"],
        "Dice selecionado": SUMMARY["selection"]["metrics_selected"]["macro"]["dice"]["mean"],
    }
)
display(SAFETY)
for name in (
    "first_training_batch.png",
    "training_curves.png",
    "threshold_curves.png",
    "fixed_validation_panel.png",
):
    display(Image.open(OUT / name))

## 9. Verificação dos arquivos da execução

Esta célula apenas lista os arquivos exigidos; não escolhe o resultado.

In [ ]:
required = [
    "final_summary.json",
    "history.csv",
    "training_curves.png",
    "threshold_curve.csv",
    "threshold_curves.png",
    "threshold_selection.json",
    "fixed_validation_panel.png",
    "validation_by_area_bin.csv",
    "validation_per_image_selected.csv",
    "validation_per_image_at_0_5.csv",
    "validation_metrics_selected.json",
    "validation_metrics_at_0_5.json",
    "augmentation_safety.json",
    "first_training_batch.png",
    "environment_report.json",
    "config_used.yaml",
]
missing = [name for name in required if not (OUT / name).is_file()]
display(
    pd.DataFrame({"arquivo": required, "presente": [(OUT / name).is_file() for name in required]})
)
assert not missing, "Arquivos ausentes: " + str(missing)
print("Arquivos da execução verificados:", OUT)

## Artefatos da execução

Arquive os resultados listados ou compacte o diretório da execução. A comparação usa exclusivamente a validação do DFUC2022.